<a href="https://colab.research.google.com/github/mausamrijall/autoshorts/blob/main/colab/AutoShorts_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 AutoShorts — Google Colab Pipeline (GPU)

**AutoShorts** turns long-form video/audio into viral **9:16 vertical Shorts** with AI-powered moment ranking.
This notebook runs the **full pipeline on Google Colab's free NVIDIA GPU** — local Whisper transcription is GPU-accelerated (no cloud transcription API needed), and clip rendering uses GPU NVENC encoding when available.

- 📦 Repo: [mausamrijall/autoshorts](https://github.com/mausamrijall/autoshorts)
- 🔁 Pipeline: `ingest → extract audio → transcribe (GPU Whisper) → LLM moment detection → render vertical clips`
- 🐍 `colab/autoshorts_pipeline.py` is a faithful Python port of the desktop app's Rust backend (`media.rs`, `transcription.rs`, `llm.rs`).

> 💡 **Recommended setup:** use **Gemini** for moment detection (free API key from [Google AI Studio](https://aistudio.google.com/apikey)) and local GPU Whisper for transcription — so your only cost is $0. You can also run fully offline with local **Ollama** models (optional cell below).


In [81]:
# ── Configuration & Input settings ───────────────────────────────
YOUTUBE_URL = "https://www.youtube.com/watch?v=TjPPIf-M6KE"

# ── Transcription ────────────────────────────────────────────────
TRANSCRIPTION_MODE = "local"      # GPU Whisper, no API key needed
WHISPER_MODEL = "base"

# ── Moment detection (LLM) ────────────────────────────────────────
LLM_PROVIDER = "openai_compat"
OPENAI_COMPAT_BASE_URL = "https://api.cohere.com/v2"
OPENAI_COMPAT_MODEL = "command-r-plus-08-2024"
COHERE_API_KEY = "cohere_Ikn1hre90exTggGPbsNNf2C4mH9RnQok3ngb7OPW4Vv635"

# Fallback / optional model variables to prevent NameErrors in Step 3
DEEPSEEK_MODEL = None
ANTHROPIC_MODEL = None
GEMINI_MODEL = None
OPENAI_MODEL = None
OPENROUTER_MODEL = None
GROQ_MODEL = None

# ── Rendering ─────────────────────────────────────────────────────
NUM_CLIPS = 5
CAPTION_STYLE = "modern-box"
RENDER_WITH_CAPTIONS = True

In [82]:
# ── Environment & GPU check ───────────────────────────────────────
import os, sys, shutil, platform, subprocess

print("Python:", platform.python_version(), "| machine:", platform.machine())
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "[WARN] no NVIDIA GPU detected — Whisper will fall back to CPU"

# System dependency (ffmpeg/ffprobe for audio extraction + clip rendering)
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("ffmpeg:", shutil.which("ffmpeg"), "| ffprobe:", shutil.which("ffprobe"))

# Python deps. PyTorch+CUDA is already pre-installed in Colab, so openai-whisper
# will use the existing CUDA-enabled torch.
!pip install -q yt-dlp openai-whisper requests
print("deps installed")


Python: 3.13.16 | machine: x86_64
Tesla T4, 15360 MiB
ffmpeg: /usr/bin/ffmpeg | ffprobe: /usr/bin/ffprobe
deps installed


In [83]:
# ── Clone AutoShorts from GitHub ───────────────────────────────────
REPO_URL = "https://github.com/mausamrijall/autoshorts.git"
REPO_DIR = "/content/autoshorts"
BRANCH = "main"

def _sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip() or f"command failed: {cmd}")
    return r

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    _sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    _sh(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH])

# Make the pipeline module importable
sys.path.insert(0, os.path.join(REPO_DIR, "colab"))
print("cloned to", REPO_DIR)


cloned to /content/autoshorts


In [84]:
# ── API keys (optional depending on provider) ──────────────────────
import getpass
try:
    from google.colab import userdata
except ImportError:
    userdata = None

def _secret(name: str):
    if userdata is not None:
        try:
            return userdata.get(name)
        except Exception:
            return None
    return None

API_KEYS = {
    "DEEPGRAM_API_KEY": _secret("DEEPGRAM_API_KEY"),
    "DEEPSEEK_API_KEY": _secret("DEEPSEEK_API_KEY"),
    "ANTHROPIC_API_KEY": _secret("ANTHROPIC_API_KEY"),
    "GEMINI_API_KEY": _secret("GEMINI_API_KEY"),
    "OPENAI_API_KEY": _secret("OPENAI_API_KEY"),
    "GROQ_API_KEY": _secret("GROQ_API_KEY"),
    "OPENROUTER_API_KEY": _secret("OPENROUTER_API_KEY"),
    "OPENAI_COMPAT_API_KEY": _secret("OPENAI_COMPAT_API_KEY"),
}

_KEY_FOR_PROVIDER = {
    "deepgram": "DEEPGRAM_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "openai": "OPENAI_API_KEY",
    "groq": "GROQ_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "openai_compat": "OPENAI_COMPAT_API_KEY",
}

needed = set()
if TRANSCRIPTION_MODE == "deepgram":
    needed.add(_KEY_FOR_PROVIDER["deepgram"])
if LLM_PROVIDER != "ollama":
    needed.add(_KEY_FOR_PROVIDER[LLM_PROVIDER])

for key in needed:
    if not API_KEYS.get(key):
        API_KEYS[key] = getpass.getpass(f"Paste {key}: ")

for key in sorted(needed):
    print(f"{key}: {'✓ set' if API_KEYS.get(key) else '✗ missing'}")


Paste OPENAI_COMPAT_API_KEY: ··········
OPENAI_COMPAT_API_KEY: ✓ set


In [85]:
# ── Input media: YouTube URL OR file upload ─────────────────────────
import pathlib, shutil, time, json

WORK = pathlib.Path("/content/autoshorts_work")
SOURCE_DIR = WORK / "source"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

if YOUTUBE_URL.strip():
    print("Downloading YouTube video …")
    _sh([
        "yt-dlp", "-f", "bv*+ba/b", "--merge-output-format", "mp4",
        "-o", f"{SOURCE_DIR}/input.%(ext)s", YOUTUBE_URL,
    ])
else:
    from google.colab import files
    uploaded = files.upload()          # pick your video/audio file
    for name, data in uploaded.items():
        (SOURCE_DIR / name).write_bytes(data)

media_files = sorted(SOURCE_DIR.iterdir())
assert media_files, "No media file found. Upload one above or set YOUTUBE_URL."
SOURCE_PATH = str(media_files[0])
print("Source media:", SOURCE_PATH)

import autoshorts_pipeline as ap


Source media: /content/autoshorts_work/source/input.mp4


In [88]:
import subprocess
import json

# Get the video title using yt-dlp without re-downloading
try:
    res = subprocess.run(
        ["yt-dlp", "--get-title", YOUTUBE_URL],
        capture_output=True, text=True, check=True
    )
    video_title = res.stdout.strip()
    print(f"🎬 YouTube Video Title: {video_title}")
except Exception as e:
    print(f"Could not retrieve title directly: {e}")

🎬 YouTube Video Title: Sam Altman On AI Revolution, Superintelligence & Future Of Humanity | FO583 Raj Shamani


In [89]:
# ── (Optional) Local Ollama with GPU for fully-offline detection ──
# Only runs when LLM_PROVIDER == "ollama". Uses the Colab GPU via CUDA.
if LLM_PROVIDER == "ollama":
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(8)
    _sh(["ollama", "pull", OLLAMA_MODEL])


In [90]:
# ── Step 1–2: probe media + extract audio + transcribe (GPU) ───────
probe = ap.probe_media(SOURCE_PATH)
print(f"input: {probe['width']}x{probe['height']}, duration={probe['duration_sec']:.1f}s, video={probe['has_video']}")

audio_path = ap.extract_audio(SOURCE_PATH, WORK / "project")
print("audio:", audio_path)

if TRANSCRIPTION_MODE == "deepgram":
    transcript = ap.transcribe_deepgram(audio_path, API_KEYS["DEEPGRAM_API_KEY"])
    engine = "Deepgram cloud"
else:
    import torch
    transcript = ap.transcribe_local(audio_path, model_size=WHISPER_MODEL)
    device = "CUDA" if torch.cuda.is_available() else "CPU"
    engine = f"Whisper/{WHISPER_MODEL} on {device}"

print(f"transcribed with {engine}: {len(transcript['words'])} words, {len(transcript['segments'])} segments")


input: 3840x2160, duration=1999.0s, video=True
audio: /content/autoshorts_work/project/transcription_audio.wav
transcribed with Whisper/base on CUDA: 5587 words, 492 segments


In [93]:
import importlib
import json
import requests
import time
import autoshorts_pipeline as ap
importlib.reload(ap)

# Dynamically locate helper functions inside the pipeline module
prompt_builder = None
for attr in dir(ap):
    if "prompt" in attr.lower() and callable(getattr(ap, attr)):
        prompt_builder = getattr(ap, attr)
        break

parser = None
for attr in dir(ap):
    if "parse_candidate" in attr.lower() and callable(getattr(ap, attr)):
        parser = getattr(ap, attr)
        break

def split_transcript_by_duration(transcript, chunk_duration_sec=300.0):
    """Splits a long transcript into chunks of fixed duration."""
    chunks = []
    current_words = []
    current_segments = []
    current_start = 0.0

    for seg in transcript.get("segments", []):
        if seg["end"] - current_start > chunk_duration_sec and current_words:
            chunks.append({
                "language": transcript.get("language", "en"),
                "duration": seg["start"] - current_start,
                "words": current_words,
                "segments": current_segments
            })
            current_words = []
            current_segments = []
            current_start = seg["start"]

        current_segments.append(seg)
        seg_start = seg["start"]
        seg_end = seg["end"]
        # Gather words that belong to this segment
        for word in transcript.get("words", []):
            if word["start"] >= seg_start and word["end"] <= seg_end:
                current_words.append(word)

    if current_words:
        chunks.append({
            "language": transcript.get("language", "en"),
            "duration": transcript["duration"] - current_start,
            "words": current_words,
            "segments": current_segments
        })
    return chunks

def call_cohere_v2_api(sub_transcript, api_key, model_name, chunk_idx):
    # Force minimum duration to be at least 15.0 seconds
    min_dur = 15.0

    # OPTIMIZATION: Create a lightweight version of the chunk for the LLM payload.
    light_transcript = {
        "language": sub_transcript.get("language", "en"),
        "duration": sub_transcript.get("duration", 0.0),
        "segments": [
            {
                "text": seg["text"],
                "start": round(seg["start"], 2),
                "end": round(seg["end"], 2)
            }
            for seg in sub_transcript.get("segments", [])
        ]
    }

    if prompt_builder:
        try:
            prompt = prompt_builder(light_transcript, min_dur)
        except TypeError:
            try:
                prompt = prompt_builder(light_transcript)
            except Exception:
                prompt = ""
    else:
        prompt = f"Analyze this transcript chunk and find viral segments between 15 and 60 seconds:\n{json.dumps(light_transcript)}"

    url = "https://api.cohere.com/v2/chat"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    # Enforce strict criteria for strong openers and clean closures
    system_instruction = (
        "You are a viral video strategist. Your job is to find high-impact viral clips from the transcript.\n"
        "CRITICAL CRITERIA:\n"
        "1. EVERY clip must be strictly MORE than 15.0 seconds long (minimum 15s, maximum 60s).\n"
        "2. Each clip MUST have a nice, compelling opener (a clear hook or logical start of an idea).\n"
        "3. Each clip MUST have a nice, natural closer (a satisfying wrap-up sentence or punchline, not cut off mid-sentence).\n"
        "4. You must respond strictly with a valid JSON array of objects representing viral clip candidates containing start, end, score, hook, and rationale. Do not wrap your response in markdown code blocks."
    )

    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "system",
                "content": system_instruction
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    print(f"\n=== [Chunk {chunk_idx}] Prompt Preview (First 500 chars) ===")
    print(prompt[:500] + "...")
    print("===================================================")

    print(f"  -> Sending optimized chunk {chunk_idx} request to Cohere API...")
    start_time = time.time()

    try:
        resp = requests.post(url, json=payload, headers=headers, timeout=60)
    except requests.exceptions.Timeout:
        print(f"  [Warning] Chunk {chunk_idx} timed out after {time.time() - start_time:.1f}s!")
        return []
    except Exception as e:
        print(f"  [Warning] Chunk {chunk_idx} encountered error: {e}")
        return []

    print(f"  -> Chunk {chunk_idx} response received in {time.time() - start_time:.1f}s (Status: {resp.status_code})")

    print(f"\n=== [Chunk {chunk_idx}] Raw Response ===")
    print(resp.text[:1000] + "..." if len(resp.text) > 1000 else resp.text)
    print("========================================")

    if resp.status_code != 200:
        return []

    resp_json = resp.json()
    text = ""
    try:
        text = resp_json.get("message", {}).get("content", [{}])[0].get("text", "")
    except Exception:
        text = ""

    raw_candidates = []
    if parser:
        try:
            raw_candidates = parser(text, min_dur)
        except Exception:
            pass
    if not raw_candidates:
        try:
            raw_candidates = json.loads(text)
        except Exception:
            raw_candidates = []

    # Client-side validation: Keep only candidates >= 15 seconds
    filtered_candidates = []
    for cand in raw_candidates:
        dur = cand.get("end", 0.0) - cand.get("start", 0.0)
        if dur >= 15.0:
            filtered_candidates.append(cand)

    return filtered_candidates

# ── Step 3: LLM viral-moment detection ────────────────────────────
api_key = COHERE_API_KEY if 'COHERE_API_KEY' in globals() else API_KEYS.get("OPENAI_COMPAT_API_KEY")
model = OPENAI_COMPAT_MODEL if OPENAI_COMPAT_MODEL else "command-r-plus-08-2024"

chunks = split_transcript_by_duration(transcript, chunk_duration_sec=300.0)
print(f"Split transcript into {len(chunks)} temporal chunks for processing...")

candidates = []
for idx, chunk in enumerate(chunks, 1):
    print(f"\n[Chunk {idx}/{len(chunks)}] duration={chunk['duration']:.1f}s | Start extraction...")
    chunk_candidates = call_cohere_v2_api(chunk, api_key, model, idx)
    print(f"-> Found {len(chunk_candidates)} candidates matching criteria (>=15s) in chunk {idx}")
    candidates.extend(chunk_candidates)

print(f"\nDetected {len(candidates)} total clip candidates across all chunks")

if candidates:
    import pandas as pd
    df = pd.DataFrame(candidates)
    df["duration_s"] = (df["end"] - df["start"]).round(1)
    df = df[df["duration_s"] >= 15.0]
    df = df[["start", "end", "duration_s", "score", "hook", "rationale"]]
    df.columns = ["Start (s)", "End (s)", "Duration (s)", "Score", "Hook", "Rationale"]
    from IPython.display import display
    display(df.sort_values(by="Score", ascending=False))
else:
    print("No clip candidates detected.")

Split transcript into 7 temporal chunks for processing...

[Chunk 1/7] duration=300.2s | Start extraction...

=== [Chunk 1] Prompt Preview (First 500 chars) ===
Analyze this transcript chunk and find viral segments between 15 and 60 seconds:
{"language": "en", "duration": 300.24, "segments": [{"text": "People are secretly making thousands of dollars using Claude to make AI influencers like this, this, and this.", "start": 0.0, "end": 6.3}, {"text": "Often the comfort of their own home without showing their face.", "start": 6.66, "end": 9.4}, {"text": "I've generated $100,000 in the last 30 days with my AI creators.", "start": 9.52, "end": 13.34}, {"text...
  -> Sending optimized chunk 1 request to Cohere API...
  -> Chunk 1 response received in 17.6s (Status: 200)

=== [Chunk 1] Raw Response ===
{"id":"232d904d-5498-4e64-926a-4c96377359fd","message":{"role":"assistant","content":[{"type":"text","text":"[\n    {\n        \"start\": 0.0,\n        \"end\": 28.28,\n        \"score\": 0.8,\

KeyboardInterrupt: 

In [ ]:
# ── Step 4: render top-N vertical clips + preview + download ───────
from IPython.display import Video
CLIPS_DIR = WORK / "clips"
CLIPS_DIR.mkdir(exist_ok=True)
crop_w = ap._cropped_width(probe)

for rank, cand in enumerate(candidates[:NUM_CLIPS], start=1):
    dst = CLIPS_DIR / f"clip-{rank:02d}_flat.mp4"
    drawtext = None
    if RENDER_WITH_CAPTIONS:
        drawtext = ap.build_drawtext_filters(
            transcript["words"], cand["start"], cand["end"], crop_w, CAPTION_STYLE
        )
    try:
        ap.render_flat_clip(SOURCE_PATH, cand["start"], cand["end"], dst, drawtext, gpu_encode=True)
    except Exception as exc:
        print(f"[warn] captions failed ({exc}); rendering clean clip")
        ap.render_flat_clip(SOURCE_PATH, cand["start"], cand["end"], dst, gpu_encode=True)

    print(f"rendered {dst.name}  [{cand['start']:.1f}s – {cand['end']:.1f}s]  score={cand['score']:.2f}")
    display(Video(str(dst), embed=True, width=270))

zip_path = "/content/autoshorts_clips.zip"
shutil.make_archive(zip_path[:-4], "zip", CLIPS_DIR)
print("\nAll clips zipped →", zip_path)
from google.colab import files
files.download(zip_path)


## 📋 Notes & troubleshooting

- **GPU**: Colab assigns a GPU (e.g. T4) via *Runtime → Change runtime type → Hardware accelerator → GPU*. Without a GPU, Whisper falls back to CPU (slower but works).
- **GPU rendering**: clips are encoded on the GPU via **NVENC** (`h264_nvenc`) with automatic CPU `libx264` fallback.
- **API keys**: For cloud providers, add them as Colab *Secrets* (the 🔑 key icon in the left sidebar) or paste them when prompted. Recommended: **Gemini** — grab a free key at [Google AI Studio](https://aistudio.google.com/apikey) and add it as `GEMINI_API_KEY` (default model `gemini-2.5-flash`; override with `GEMINI_MODEL`). [DeepSeek](https://platform.deepseek.com) is a cheap alternative.
- **Custom provider**: set `LLM_PROVIDER = "openai_compat"` and `OPENAI_COMPAT_BASE_URL` to use any OpenAI-compatible proxy/gateway (e.g. OpenRouter, LiteLLM, your own backend).
- **Fully offline**: set `LLM_PROVIDER = "ollama"` — the optional Ollama cell installs a local model on the GPU, so no API keys are needed at all.
- **Whisper model sizes**: `tiny`/`base` are fast; `small`/`medium`/`large-v3` are more accurate but slower. If you change `WHISPER_MODEL` mid-session, restart the runtime (the model is cached in memory).
- **Outputs** live in `/content/autoshorts_work/` (clips + transcript JSON) until the runtime ends — download the ZIP before closing.
- **Captions**: if ffmpeg's `drawtext` fails (rare font issues), clips render clean without captions.
- **9:16 crop**: landscape input is center-cropped to a vertical strip, exactly like the desktop app.
